# Data Visualisation et Analyse de Cohortes sur Olist.

### **_Un peu de contexte..._**

**La plateforme e-commerce Olist connecte les propriétaires de petits commerces brésiliens (les vendeurs) avec des consommateurs partout au Brésil.** Comprendre ce qui pousse les vendeurs à continuer d'utiliser la plateforme une fois qu'ils ont effectué leur première vente est une **question essentielle** pour Olist !

**L'analyse de cohortes vise à regrouper les individus en 'cohortes' selon une caractéristique commune, puis à étudier l'évolution de ces cohortes au fil du temps.**

Un petit exemple pour comprendre cela :

> _Il est possible d'étudier l'impact d'une **campagne marketing** sur différentes cohortes, d'analyser le taux d'attrition des cohortes, etc._

### **_La Mission..._**

**Olist vous a engagé comme Data Analyst pour réaliser une analyse de cohortes sur leurs données 2017-2018.**

Ici, les clients d'Olist sont les vendeurs du site. Il s'agit de les répartir en cohortes correspondant au mois de leur première vente. Il ne s'agit pas tout à fait d'une analyse de cohortes classique : ici, l'objectif sera de suivre la **proportion de vendeurs de chaque cohorte qui continue à utiliser la plateforme** dans les mois suivants leur première vente.

### **_Les données à analyser..._**

Les données à utiliser sont disponibles sur la plateforme ainsi que sur [Kaggle](https://www.kaggle.com/olistbr/brazilian-ecommerce). Nous utiliserons les tables `olist_sellers_dataset`, `olist_orders_dataset` et `olist_order_items_dataset`.

> _N'hésitez pas à consulter la documentation sur les tables disponible sur Kaggle : [sur ce lien](https://www.kaggle.com/olistbr/brazilian-ecommerce)._


### On commence par importer les packages nécéssaires.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

###  Fusionner les tables à analyser.

Pour cette analyse, nous n'aurons besoin que de **deux variables :**

- Les **IDs** des vendeurs dans `olist_sellers_dataset`.
- La **date** de chaque vente (`order_purchase_timestamp`) dans `olist_orders_dataset`.

**Fusionner les tables et ne conserver que les 2 colonnes pertinentes.**

Pour connecter les deux tables, il est indispensable d'utiliser `olist_order_items_dataset` comme intermédiaire (avec `order_id`).

#### Les tables sont importées via un repository GitHub, exécuter la cellule ci-dessous :

In [ ]:
df_orders = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_orders_dataset.csv')
df_order_items = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_items_dataset.csv')
df_sellers = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_sellers_dataset.csv')

In [ ]:
df_order_items.info()

In [ ]:
df_order_items

In [ ]:
df_orders.info()

In [ ]:
df_sellers.info()

**Maintenant que les tables sont importées dans le notebook, il va falloir joindre les données.**

In [ ]:
df = df_order_items.merge(df_orders, on='order_id', how='inner')
df = df.merge(df_sellers, on='seller_id', how='inner')
df.info()

In [ ]:
data = df[['seller_id', 'order_purchase_timestamp']]

###  Filtrer les dates.


**Convertir la colonne de dates au bon type, et conserver uniquement les données de Jan 2017 à Sept 2018.**

Il est possible (mais optionnel) d'utiliser les méthodes `df.assign()` et `df.query()`. La méthode `df.assign()` permet de créer une nouvelle colonne à partir d'une fonction lambda. `df.query()` permet de filtrer un dataframe selon la date.

**Ces méthodes sont bonus, elles permettent d'être plus concis mais ne sont pas du tout obligatoires.**

In [ ]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_purchase_timestamp']

In [ ]:
data['date'] = pd.to_datetime(data['order_purchase_timestamp']) # Création d'une colonne 'date' du type datetime.

# Nous pouvons maintenant supprimer la colonne 'order_purchase_timestamp'.
data = data.drop('order_purchase_timestamp', axis=1)

data = data[ (data['date'] >= '2017') & (data['date'] <= '09-2018') ] # Conditions de filtrage qui permettent de filtrer le dataframe de Jan 2017 à Sept 2018.
display(data)

In [ ]:
data.info()

In [ ]:
df.sort_values(by='order_purchase_timestamp', ascending=False) ## ranger par date du plus grand au plus petit

In [ ]:
Start_date = '2017-01-01'
End_date = '2018-09-30'

In [ ]:
df_1 = df[(df['order_purchase_timestamp'] > Start_date) & (df['order_purchase_timestamp'] < End_date)]
df_1 = df_1.sort_values(by='order_purchase_timestamp', ascending=True)

In [ ]:
df_1 = df_1.assign(order_year_month=lambda x: x['order_purchase_timestamp'].dt.to_period('M'))

In [ ]:
## df_1 = df_1.drop(columns=['order_purchase_timestamp'])

In [ ]:
df.query('order_purchase_timestamp >= "2017-01-01" and order_purchase_timestamp <= "2018-09-30"') ## avec query

### Constitution des cohortes.


#### **Attention, les cohortes doivent être définies selon le mois de la première vente, c'est-à-dire, une cohorte par mois.**

Pour créer les cohortes, commencer par **extraire** le mois de vente à partir des **dates complètes.**

Ensuite, ordonner les données selon le **mois de vente** et **supprimer les doublons** dans la colonne `seller_id`, afin de ne conserver que la première apparition de chaque vendeur...

> _Pour extraire la période annee.mois, on peut utiliser la méthode `df['col'].dt.to_period('M')`._

In [ ]:
data_vendeur = data_vendeur.sort_values(by='date')
data_vendeur = data_vendeur.drop_duplicates(subset=['seller_id'], keep='first')
data_vendeur = data_vendeur.assign(date_year_month=lambda x: x['date'].dt.to_period('M'))
data_vendeur = data_vendeur.reset_index().drop('index', axis=1).drop(columns= 'date')
data_vendeur

In [ ]:
## df_2 = df_1.drop_duplicates(subset=['seller_id'])

### Visualiser le nombre de vendeurs dans chaque cohorte grâce à un diagramme en bâtons.

Avant de continuer l'analyse, nous allons **visualiser nos données**. En particulier, nous aimerions avoir une idée de la taille de chaque cohorte.

**Nous allons compter le nombre de vendeurs uniques dans chaque cohorte, puis utiliser seaborn pour créer un bar chart.**

_Souvenez-vous de la fonction `sns.catplot()` !_

In [ ]:
.groupby()

In [ ]:
data_vendeur.groupby('date_year_month').sum()

In [ ]:
data_vendeur['seller_id'].nunique()

In [ ]:
cohorte = data_vendeur.groupby('date_year_month')['seller_id'].count().reset_index()
cohorte

In [ ]:
## nbr_vendeur = df_2.loc['seller_id']
##nbr_vendeur
##df_2.groupby('order_year_month').count()

In [ ]:
sns.catplot(data=cohorte, x='date_year_month', y='seller_id', kind='bar', aspect =2 , height = 8) ## correction ben

In [ ]:
sns.catplot(data=df_2, x='order_year_month', kind='count', aspect =2 , height = 8)

In [ ]:
df_2

In [ ]:
df_2.count()

In [ ]:
cohorte_vendeur = df_2[['seller_id', 'order_year_month']]

cohorte_vendeur = cohorte_vendeur.sort_values(by='order_year_month', ascending=True)

In [ ]:
test['seller_id'].unique()

In [ ]:
test = df_2[['seller_id', 'order_year_month']]
display(test)

On observe une **certaine variabilité** dans le nombre de nouveaux vendeurs par mois, mais aucun **effet saisonnier** ne semble évident.

#### **Étudions l'attrition de ces cohortes.**

### Pour chaque vendeur.

- Calculer **la différence** entre chacune des **ventes** et sa **cohorte** (mois de première vente).

- **Fusionner la table** contenant les **noms** de cohortes (question précédente) avec la **table** initiale (contenant toutes les ventes).

> _Il est maintenant **possible de calculer** la différence (en mois) entre la date de chaque vente et la cohorte du vendeur (mois de première vente)._

#### **Vous pourrez appeler cette différence `delta`.**

In [ ]:
cohorte

In [ ]:
df_2.index.name = 'ID'
df_2

In [ ]:
diff['Delta'] = df_2['order_purchase_timestamp'].dt.month - df_2['order_year_month'].dt.month

###  Étude de l'attrition des vendeurs, étude du Churn.

- Pour chaque cohorte, **compter le nombre de vendeurs uniques par delta.**

- **Convertir ce nombre** pour l'exprimer en **pourcentage** de la taille de la cohorte.

- Ces calculs nous **permettent** d'observer quelle proportion de chaque cohorte continue de **réaliser des ventes** au fil du temps.

> _Il faudra d'abord **grouper votre table par cohorte** et par delta pour obtenir le nombre de vendeurs uniques par cohorte et par delta. Ensuite, **on pourra fusionner cette table** avec les totaux calculés dans la question précédente, afin de calculer les pourcentages par rapport au total de chaque cohorte._

### Pivoter la table.

Utiliser  `df.pivot()` (ou `df.pivot_table`) pour **obtenir une table avec les valeurs de delta** en colonnes, les différentes cohortes en lignes et les proportions calculées comme valeurs.


### Visualiser les cohortes finales et les taux d'attrition grâce à une heatmap.

> _Vous pourrez utiliser `plt.figure(figsize=(12, 8))` en première ligne de votre cellule pour ajuster la taille du graphique. Vous pourrez **également** ajouter dans la fonction `sns.heatmap()` les arguments : `cmap='crest', annot=True, fmt='.0f'`, pour un résultat **plus esthétique !**_

# Bonus : Différents types de segmentation.

### Commençons par un pré-traitement des données.

Commencez par **exécuter les cellules** ci-dessous pour charger les données, **fusionner les tables** et effectuer toutes les étapes de pré-traitement.

> _Ci-dessous, les données sont lues directement depuis GitHub pour éviter le téléchargement. La fonction `clean_olist` permet de réaliser le pré-traitement en une seule ligne (vous pouvez cependant tester chaque étape du pré-traitement de votre côté, ce qui pourrait déjà être un bon exercice en soi)._

#### **Attention, même avec la fonction, il reste toujours quelques imperfections sur lesquelles vous pourrez vous pencher.**

In [ ]:
# Chargement des tables :

customers = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_customers_dataset.csv')
items = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_items_dataset.csv')
payments = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_reviews_dataset.csv')
orders = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_orders_dataset.csv')
products = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_products_dataset.csv')
sellers = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_sellers_dataset.csv')
names = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/product_category_name_translation.csv')

In [ ]:
def clean_olist(customers, items, payments, reviews, orders, products, sellers, names, merge_all=True):
    """
    Fonction de nettoyage et de transformation des différentes tables opérationnelles Olist.
    Si merge_all = True, la fonction renvoie une seule dataframe prête pour l'analyse de cohorte.
    Si merge_all = False, la fonction renvoie 4 dataframes distinctes à fusionner manuellement.
    """

    # Fusion des différentes tables entre elles
    merged = items.merge(products, on='product_id', how='outer')\
                   .merge(sellers, on='seller_id', how='outer')\
                   .merge(orders, on='order_id', how='outer')\
                   .merge(payments, on='order_id', how='outer')\
                   .merge(reviews, on='order_id', how='outer')\
                   .merge(customers, on='customer_id', how='outer')

    # Suppression des colonnes non pertinentes pour l'analyse
    merged = merged.drop(columns=['payment_sequential',
                                  'customer_id',  # Identifiant client non pertinent pour l'analyse
                                  'customer_zip_code_prefix', 'customer_city',
                                  'seller_zip_code_prefix', 'seller_city',  # Données redondantes
                                  'review_id', 'review_comment_title', 'review_comment_message',
                                  'review_creation_date', 'review_answer_timestamp',  # Les commentaires ne sont pas analysés
                                  'order_approved_at', 'order_delivered_carrier_date', 'shipping_limit_date',
                                  'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                                  'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'])  # Colonnes non pertinentes

    # Calcul du délai de livraison réel par rapport à l'estimation
    merged['delivery_delay'] = pd.to_datetime(merged.order_delivered_customer_date) - pd.to_datetime(merged.order_estimated_delivery_date)
    merged['delivery_delay'] = merged['delivery_delay'].dt.round('d')  # Arrondir à des jours entiers
    merged = merged.drop(columns=['order_delivered_customer_date', 'order_estimated_delivery_date'])  # Supprimer les colonnes utilisées pour le calcul

    # Calcul de la quantité totale de chaque produit par commande
    quantity_table = merged[['order_id', 'order_item_id']].groupby('order_id').max().reset_index()
    quantity_table.columns = ['order_id', 'order_item_quantity']  # Renommer les colonnes
    merged = merged.merge(quantity_table, on='order_id', how='outer')  # Fusionner les données calculées

    # Suppression des doublons liés aux paiements
    merged = merged.drop_duplicates(subset=['order_id', 'payment_value'])

    # Calcul du nombre maximum d'installments (paiements échelonnés) pour chaque commande
    installments_table = merged[['order_id', 'payment_installments']].groupby('order_id').max().reset_index()
    installments_table.columns = ['order_id', 'payment_instalments']  # Renommer les colonnes
    merged = merged.merge(installments_table, on='order_id', how='outer')  # Fusionner les données

    # Calcul du nombre de types de paiements utilisés par commande
    types_table = merged[['order_id', 'payment_type']].groupby('order_id').nunique().reset_index()
    types_table.columns = ['order_id', 'payment_types']  # Renommer les colonnes
    merged = merged.merge(types_table, on='order_id', how='outer')

    # Somme des valeurs des paiements pour chaque commande
    payment_table = merged[['order_id', 'payment_value']].groupby('order_id').sum().reset_index()
    payment_table.columns = ['order_id', 'total_payment_value']  # Renommer les colonnes
    merged = merged.merge(payment_table, on='order_id', how='outer')

    # Suppression des doublons de commande et de produit
    merged = merged.drop_duplicates(subset=['order_id', 'product_id'])

    # Suppression des anciennes colonnes non nécessaires
    merged = merged.drop(columns=['order_item_id', 'payment_type', 'payment_installments', 'payment_value'])

    # Calcul du prix total par produit par commande (incluant la quantité)
    merged['order_item_value'] = merged.order_item_quantity * (merged.price + merged.freight_value)

    # Suppression des valeurs manquantes sur les identifiants produit
    merged = merged[~merged['product_id'].isna()].reset_index(drop=True)

    # Ajout du préfixe 'BR-' aux états pour indiquer le pays (Brésil)
    merged.customer_state = merged.customer_state.apply(lambda x: 'BR-' + x)
    merged.seller_state = merged.seller_state.apply(lambda x: 'BR-' + x)

    # Traduction des catégories de produits du portugais vers l'anglais
    portuguese = list(list(names.to_dict().values())[0].values())  # Catégories en portugais
    english = list(list(names.to_dict().values())[1].values())  # Catégories traduites en anglais
    language_pairs = {k: v for k, v in zip(portuguese, english)}
    language_pairs['nan'] = np.nan
    language_pairs['pc_gamer'] = 'pc_gamer'
    language_pairs['portateis_cozinha_e_preparadores_de_alimentos'] = 'portable_kitchen_food_preparers'
    merged['product_category_name'] = merged['product_category_name'].apply(lambda x: language_pairs[str(x)])

    # Si l'option merge_all est activée, renvoyer une dataframe unique avec les colonnes pertinentes
    if merge_all:
        return merged[['order_id', 'customer_unique_id', 'seller_id', 'product_id', 'product_category_name',
                       'price', 'freight_value', 'order_item_quantity', 'order_item_value',
                       'total_payment_value', 'payment_types', 'payment_instalments', 'review_score',
                       'order_purchase_timestamp', 'order_status', 'delivery_delay',
                       'customer_state', 'seller_state']]

    else:
        # Si merge_all est désactivé, renvoyer 4 dataframes séparées
        orders_info = merged[['order_id', 'customer_unique_id', 'seller_id', 'product_id',
                              'price', 'freight_value', 'order_item_quantity', 'order_item_value',
                              'total_payment_value', 'payment_types', 'payment_instalments', 'review_score',
                              'order_purchase_timestamp', 'order_status', 'delivery_delay']]
        customers_info = merged[['customer_unique_id', 'customer_state']]
        sellers_info = merged[['seller_id', 'seller_state']]
        product_info = merged[['product_id', 'product_category_name']]
        return (orders_info, product_info, customers_info, sellers_info)

# Utilité de la fonction `clean_olist()`

Comme nous l'avons constaté, la fonction `clean_olist` est conçue pour **nettoyer** et **transformer** plusieurs **tables de données** liées à la plateforme Olist (clients, commandes, produits, paiements, etc.), en vue **d'analyses**.

> Elle fusionne ces tables sur des identifiants communs, supprime les colonnes non pertinentes pour l’analyse et calcule plusieurs indicateurs, comme les délais de livraison, le total des paiements ou la quantité de produits commandés. Elle peut également traduire les catégories de produits du portugais vers l’anglais. Selon l’option `merge_all`, elle renvoie soit une seule DataFrame consolidée pour l’analyse, soit quatre DataFrames distinctes pour des analyses spécifiques.
